
# Placing July 2026 in context: temperature, rainfall and moisture climatologies

**Notebook 3 of 3 — Chicagoland, July 2026 severe weather and flooding case study**

The first two notebooks in this series established *what happened* on 27 July 2026 (seven confirmed
tornadoes in a DuPage-to-southern-Cook corridor between 11:54 am and 12:59 pm CDT, followed by a
bowing wind line with an 82 kt gust at Beach Park) and *why the atmosphere allowed it* (an
extraordinary co-location of tropical moisture, 4350 J kg⁻¹ of mixed-layer CAPE and 65 kt of deep-layer
shear). This notebook asks the third question, the one a radio audience actually cares about:

> **Was this a normal July? And if not, in what specific way was it abnormal?**

That question has to be answered carefully, because the honest answer is more interesting than the
easy one. July 2026 in Chicago was hot, and it was wet — but the *way* it was wet is the story. It
did not rain often; it rained hard. Six wet days delivered 5.36 inches, and 95 percent of the month's
total fell on three calendar days. Two of those days produced flooding, and one of them produced
tornadoes.

The notebook is organised as three climatologies and one synthesis:

1. **A 57-year daily temperature and precipitation climatology** at Chicago O'Hare (1970–2026), used
   to rank July 2026 against every other July in the consistent record, and to test which July metrics
   have a statistically defensible trend.
2. **A purpose-built precipitable-water climatology** from 5,025 observed July radiosondes at the three
   upper-air sites relevant to Chicagoland inflow, used to place the 27 July moisture as a percentile
   of *observed* soundings rather than against a model's own history.
3. **The 31 July – 1 August flooding event**, which turns out to be a completely different meteorological
   regime from the 27th, and whose explanation is antecedent wetness rather than rainfall rate.
4. **A synthesis** on what survives scrutiny and what does not — including an explicit correction to an
   earlier framing of the moisture anomaly.

A recurring theme is *scientific restraint*. Of ten July metrics tested for trend over 57 years, exactly
one is significant at α = 0.05. That fact is more useful to a radio interview than a list of things that
"look like they're increasing", and this notebook makes the distinction an explicit teaching point.


## Data sources and endpoints

Everything in this notebook comes from two public services, both queried over HTTP with no
authentication. Endpoints are given verbatim so the analysis is reproducible.

**Iowa Environmental Mesonet (IEM), Iowa State University** — `https://mesonet.agron.iastate.edu`

| Product | Endpoint | Used for |
|---|---|---|
| Daily climate summaries | `/cgi-bin/request/daily.py` | O'Hare daily max/min temperature, max/min dewpoint, precipitation, maximum heat index, and the station's own `climo_high_f` / `climo_low_f`; complete at ORD from 1970 |
| Radiosonde bulk archive | `/cgi-bin/request/raob.py` | Every July sounding at KILX, KDVN and KGRB, 2000–2025, one row per reported level |
| ASOS observations | `/cgi-bin/request/asos.py` | Sub-hourly precipitation, temperature, dewpoint, wind during the flood event |
| HADS / DCP river gauges | `/cgi-bin/request/hads.py` | River stage time series for the four warned gauges |
| Local storm reports | `/cgi-bin/request/gis/lsr.py` | Point rainfall totals and flood/flash-flood reports |
| Raw NWS product text | `/api/nwstext/...` | Official flood stage and forecast crest, parsed out of the NWS Flood Warning (FLW) product |

**HRRR analysis and 1-hour forecasts** via [Herbie](https://herbie.readthedocs.io), which resolves to the
NOAA HRRR archives on AWS / Google Cloud. Fields used here are `PWAT`, `MLCAPE`/`MLCIN`, 0–6 km bulk
shear, 2 m temperature and dewpoint, and 1-hour accumulated precipitation. The download scripts
(`fetch_hrrr.py`, `fetch_flood.py`) are reproduced in the companion notebooks and are not re-run here.

**Citation note.** The IEM archives are a service of the Iowa State University Department of Agronomy and
should be cited as such; the underlying observations are NWS/FAA ASOS and NWS radiosonde data. HRRR is
Dowell et al. (2022), *Weather and Forecasting* 37, 1371–1395, and Herbie is Blaylock (2022),
doi:10.5281/zenodo.4567540.

### A note on colour

Precipitation fields in this series use the `ChaseSpectral` colormap from
[**cmweather**](https://cmweather.readthedocs.io), a package of perceptually-considered colormaps built
for radar and weather fields by the Open Radar Science community. Importing `cmweather` registers its
colormaps with Matplotlib by name; no further setup is required. Precipitable water uses a diverging
brown–green ramp, which reads correctly as dry-to-moist and is safe for the most common forms of colour
vision deficiency.


## Setup

The climatology work in this notebook needs only the scientific Python stack plus `requests`;
the figures additionally use `cartopy`, `metpy` (for county boundaries and for validating the
precipitable-water integration) and `cmweather`.

In [ ]:
import io
import time
import warnings

import numpy as np
import pandas as pd
import requests
from scipy import stats

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import cmweather  # noqa: F401 - registers ChaseSpectral and friends with Matplotlib

warnings.filterwarnings("ignore")

IEM = "https://mesonet.agron.iastate.edu/cgi-bin/request"

# Figure style: a single small helper applied before every figure, so panel typography
# is consistent across the three notebooks in this series.
META_GREY = "#5a5a5a"


def apply_figure_style(*, sizes=(10, 9, 8), grid=False):
    """Publication defaults: open frames, small consistent type, white background."""
    base, secondary, tick = sizes
    plt.rcParams.update({
        "font.family": "sans-serif",
        "font.size": base,
        "axes.labelsize": base,
        "axes.titlesize": base,
        "axes.titlelocation": "left",
        "axes.titleweight": "normal",
        "legend.fontsize": secondary,
        "legend.frameon": False,
        "xtick.labelsize": tick,
        "ytick.labelsize": tick,
        "axes.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": bool(grid),
        "grid.alpha": 0.3,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.bbox": "tight",
        "savefig.dpi": 200,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
    })


def panel_letter(ax, letter, dx=-0.085, dy=1.04, fontsize=11):
    ax.text(dx, dy, letter, transform=ax.transAxes, fontsize=fontsize,
            fontweight="bold", va="bottom", ha="left")


apply_figure_style()


## 1. The O'Hare daily record, 1970–2026

### 1.1 Fetching the daily climate summaries

The IEM `daily.py` service returns one row per station-day with the quantities a monthly climatology
needs: maximum and minimum temperature, maximum and minimum dewpoint, precipitation, maximum heat
index (`max_feel`), and — usefully — the station's own published normals for that calendar day
(`climo_high_f`, `climo_low_f`). Requesting the whole 1970–2026 period for one station is a single
query returning roughly 20,000 rows.

> **Gotcha — endpoint parameters.** The query string is
> `network=IL_ASOS&stations=ORD&year1=..&month1=..&day1=..&year2=..&month2=..&day2=..&format=csv`.
> The network code matters: O'Hare lives in `IL_ASOS`, not in a national network. Missing values arrive
> as the literal strings `M` and `None`, so both must be passed to `na_values` or the temperature
> columns silently come back as `object` dtype and every subsequent `.mean()` fails or, worse, returns
> a string concatenation. The ORD record is complete from 1970; earlier years exist but with enough
> gaps that including them would bias the monthly means.

In [ ]:
url_daily = (f"{IEM}/daily.py?network=IL_ASOS&stations=ORD"
             "&year1=1970&month1=1&day1=1&year2=2026&month2=8&day2=2&format=csv")

daily = pd.read_csv(io.StringIO(requests.get(url_daily, timeout=300).text),
                    na_values=["M", "None", ""])
daily["day"] = pd.to_datetime(daily.day)
daily["yr"] = daily.day.dt.year
daily["mo"] = daily.day.dt.month

print(daily.shape, daily.day.min().date(), "->", daily.day.max().date())


### 1.2 Defining a "complete July"

A monthly ranking is only meaningful if every year in the comparison set is measured the same way. A
July missing four days of precipitation is not comparable to a complete one, and the difference shows
up exactly where it hurts — in the extremes. The screen used here requires 30 or more calendar days
present, at least 28 non-missing maximum temperatures, and at least 28 non-missing precipitation
values. That leaves **57 complete Julys** between 1970 and 2026 inclusive.

Anomalies and percentiles below are computed against the **1991–2020 normal period**, the current
WMO/NOAA convention, while ranks and percentiles use all 57 years.

In [ ]:
jul = daily[daily.mo == 7].copy()

coverage = jul.groupby("yr").agg(n=("day", "size"),
                                 tmax_n=("max_temp_f", "count"),
                                 precip_n=("precip_in", "count"))
complete = coverage[(coverage.n >= 30) & (coverage.tmax_n >= 28) & (coverage.precip_n >= 28)]

J = jul[jul.yr.isin(complete.index)].copy()
J["tavg"] = (J.max_temp_f + J.min_temp_f) / 2.0

S = J.groupby("yr").agg(
    tavg=("tavg", "mean"),
    tmax=("max_temp_f", "mean"),
    tmin=("min_temp_f", "mean"),
    tmax_abs=("max_temp_f", "max"),
    d90=("max_temp_f", lambda s: int((s >= 90).sum())),
    d95=("max_temp_f", lambda s: int((s >= 95).sum())),
    warm_nights=("min_temp_f", lambda s: int((s >= 70).sum())),
    precip=("precip_in", "sum"),
    wet_d=("precip_in", lambda s: int((s >= 0.01).sum())),
    d1in=("precip_in", lambda s: int((s >= 1.0).sum())),
    maxday=("precip_in", "max"),
    dwpt=("max_dewpoint_f", "mean"),
    feel=("max_feel", "mean"),
)

# rainfall concentration: how much of the month came from its three wettest days
conc = []
for y, g in J.groupby("yr"):
    total = g.precip_in.sum()
    wet = int((g.precip_in >= 0.01).sum())
    if total > 0.5:
        conc.append(dict(yr=y, total=total,
                         top3_share=g.precip_in.nlargest(3).sum() / total * 100,
                         wet_days=wet,
                         intensity=total / max(wet, 1)))
CC = pd.DataFrame(conc).set_index("yr")
S = S.join(CC[["top3_share", "intensity"]])

normals = S.loc[1991:2020]
n_years = len(S)
print(f"{n_years} complete Julys, {S.index.min()}-{S.index.max()}")


### 1.3 The percentile convention — define it or be misquoted

This caused genuine confusion during the analysis and is worth stating precisely, because ranks and
percentiles quoted for the same metric can look mutually inconsistent when they are not.

Two different numbers get reported for each metric:

- **rank** — the number of Julys (out of 57, *including* 2026) whose value is **greater than or equal to**
  2026's. Rank 1 means "highest on record".
- **percentile** — the **weak percentile rank**: the share of all 57 Julys whose value is **less than or
  equal to** 2026's, times 100.

For a continuous quantity with no ties these are near-reciprocal, and the tempting shortcut
`(n − rank + 1) / n` gives roughly the right answer. **For integer counts it is simply wrong**, because
many years tie. Consider the two heat-day counts in July 2026:

- **Days ≥ 90 °F**: nine days. Sixteen Julys reach nine or more, so rank = 16. But **three** years tie at
  exactly nine (1980, 2002, 2026), so only 13 years are strictly greater — 44 of 57 are less than or
  equal, giving the **77th** percentile.
- **Nights ≥ 70 °F**: twelve nights. Also rank 16. But **six** years tie at exactly twelve (1972, 1973,
  1987, 2006, 2010, 2026), so only 10 are strictly greater — 47 of 57 are less than or equal, giving the
  **82nd** percentile.

Identical rank, different percentile, and the naive formula would have returned 74th for both. The most
extreme case in this dataset is **days with ≥ 1.00 inch**: July 2026 had three, which is the record
*maximum* — no July in 57 years has exceeded it — so the weak percentile is exactly **100th** even though
four other years (1982, 2016, 2017, 2024) also reached three and the rank is therefore 5.

The function below computes both correctly, and the printed table reproduces the worked examples.

In [ ]:
def rank_and_percentile(series, year=2026):
    """Weak percentile rank and count-based rank, tie-aware.

    percentile : share of all years with value <= the target year's value, x100
    rank       : number of years with value >= the target year's value (1 = highest)
    """
    v = float(series.loc[year])
    vals = series.astype(float).values
    n = len(vals)
    pct = (vals <= v).mean() * 100.0
    rank = int((vals >= v).sum())
    ties = int((vals == v).sum())
    return dict(value=v, n=n, rank=rank, percentile=pct, ties=ties,
                naive_pct=(n - rank + 1) / n * 100.0)


LABELS = {
    "tavg": "Mean temperature (F)", "tmax": "Mean daily max (F)",
    "tmin": "Mean overnight low (F)", "d90": "Days >= 90 F",
    "d95": "Days >= 95 F", "warm_nights": "Nights >= 70 F",
    "dwpt": "Mean max dewpoint (F)", "feel": "Mean max heat index (F)",
    "precip": "Total precipitation (in)", "wet_d": "Wet days (>= 0.01 in)",
    "d1in": "Days >= 1.00 in", "intensity": "Rain per wet day (in)",
    "top3_share": "Share from wettest 3 days (%)",
}

rows = []
for col, lab in LABELS.items():
    r = rank_and_percentile(S[col])
    rows.append(dict(metric=lab, value=round(r["value"], 2),
                     anomaly=round(r["value"] - float(normals[col].mean()), 2),
                     rank=f"{r['rank']} / {r['n']}",
                     percentile=round(r["percentile"]),
                     ties=r["ties"],
                     naive=round(r["naive_pct"])))
RANKS = pd.DataFrame(rows).set_index("metric")
print(RANKS.to_string())


The `ties` and `naive` columns are diagnostic, not results: wherever `ties` exceeds 1 the naive formula
disagrees with the correct weak percentile, and the size of the disagreement is exactly the tie count.
They are printed here so a future reader can see the failure mode rather than take it on trust.

### 1.4 What the ranking says: hot, and wet in a very particular way

**Temperature.** July 2026 averaged **76.8 °F**, which is **2.2 °F above the 1991–2020 normal** and the
**13th warmest of 57** Julys — the 79th percentile. The mean daily maximum was 85.6 °F (+1.8, 74th
percentile) and the hottest day reached 96 °F. There were **9 days at or above 90 °F** (+3, 77th) and
**4 days at or above 95 °F** (+3, rank 8, 88th percentile).

The more consequential number is the overnight one. The **mean daily minimum was 68.0 °F, +2.6 °F above
normal — rank 10 of 57, 84th percentile** — and there were **12 nights at or above 70 °F** against a
normal of 8 (82nd percentile). Mean maximum dewpoint was 68.4 °F (+1.8, 75th) and the mean maximum heat
index 87.7 °F (+2.1, 70th).

> **The nights warmed more than the days: +2.6 °F versus +1.8 °F.** That asymmetry is the part with a
> public-health consequence. Heat becomes dangerous not because of the afternoon peak but because of
> failure to cool overnight: the body's recovery window closes, and in housing without air conditioning
> indoor temperature ratchets upward across consecutive days. A July whose nights run nearly 3 °F warm
> is a harder July to survive than the daytime numbers suggest.

**Precipitation.** The monthly total was **5.36 inches, 1.77 inches above normal, the 9th wettest of 57**
Julys (86th percentile) — about 1.5 times the normal monthly rainfall. Taken alone that sounds like a
persistently damp month. It was the opposite.

There were only **six wet days** (≥ 0.01 in) against a normal of 9.8. That is the **9th percentile — near
record *few***, rank 55 of 57. The month contained **three days of 1.00 inch or more** against a normal of
0.8, which **ties the 57-year record maximum** (100th percentile). Rain per wet day was **0.89 inches
versus a normal of 0.36 — the 98th percentile, second highest of 57**. And the three wettest days
delivered **95 percent of the month's total** (93rd percentile).

Those three days were **2 July (1.64 in)**, **3 July (1.71 in)** and **27 July (1.76 in — the tornado
day)**.

> **The line for the interview:** it did not rain *often* in July, it rained *hard*. A normal Chicago July
> spreads about 3.6 inches over ten days. This July dropped 5.36 inches on six days, with 95 percent of
> it in three.

That distinction is not rhetorical. A month's worth of water arriving in three pulses runs off; the same
water spread over ten days infiltrates. The concentration is the mechanism that connects the July
climatology to the flooding in Section 3.


### 1.5 Trends, and the discipline of not over-claiming

A 57-year single-station record invites trend claims, and this is where a case study most easily loses
credibility. Ten July metrics were tested with ordinary least-squares regression of the annual value on
year over 1970–2026. **Exactly one is significant at α = 0.05.**

In [ ]:
TREND_COLS = ["tavg", "tmax", "tmin", "d90", "d95", "warm_nights",
              "dwpt", "precip", "d1in", "intensity"]

trends = {}
for col in TREND_COLS:
    y = S[col].astype(float)
    trends[col] = stats.linregress(S.index.values.astype(float), y.values)

TR = pd.DataFrame([
    dict(metric=LABELS[c],
         per_decade=round(trends[c].slope * 10, 3),
         r=round(trends[c].rvalue, 3),
         p=round(trends[c].pvalue, 3),
         significant=trends[c].pvalue < 0.05)
    for c in TREND_COLS
]).set_index("metric").sort_values("p")
print(TR.to_string())


The result, stated as it should be stated on air:

**Chicago's July nights are measurably warming.** The mean daily minimum temperature rises
**+0.48 °F per decade with p = 0.022** — the only metric of the ten that clears the threshold. Over
57 years that is roughly 2.7 °F of warming in overnight lows, and it is consistent with both the
regional urban-heat-island literature and the broader observation that minimum temperatures are
warming faster than maxima across the continental interior.

**Everything else falls short.** Mean temperature trends at +0.37 °F/decade but **p = 0.079**. Total July
precipitation trends at +0.32 in/decade, **p = 0.057** — tantalising, and still not significant. Days with
≥ 1.00 inch: **p = 0.092**. Rain per wet day: **p = 0.079**. Several of these are close enough that a longer
record or a regional average would very likely resolve them, and the physical expectation
(Clausius–Clapeyron scaling of extreme rainfall with a warming lower troposphere) points the same way.
But "very likely resolves with more data" is not the same claim as "the data show it".

> **Teaching point.** The correct summary is: *July nights in Chicago are warming at a rate we can
> measure; for July rainfall and heat-day counts, a 57-year single-station record is not long enough to
> separate a trend from ordinary year-to-year variability.* That is the version that survives a hostile
> question. Saying instead that "July rainfall is increasing" is a claim this dataset does not support at
> conventional significance, and one careful listener with the same public data can take it apart.
>
> This restraint is enforced in the figures as well as the prose: **a trend line is drawn only where
> p < 0.05.** Panel (c) of Figure 1 carries a regression line; panels (a) and (b) deliberately do not,
> even though the eye wants one.

An important caveat on the test itself. This is OLS on annual values, which assumes independent years and
a linear response, and it makes no adjustment for testing ten metrics at once — under a Bonferroni
correction the surviving p-value of 0.022 would itself be marginal against a 0.005 threshold. Treat this
as a *screening* test that says where to look, not as a formal attribution study.


### 1.6 Figure 1 — July 2026 at O'Hare

The figure below assembles the ranking. Panels (a) and (b) place 2026 in the year-by-year series for mean
temperature and total precipitation, with the 1991–2020 normal as a dashed line and the normal-period
interquartile range shaded. Panel (c) is the overnight minimum — the one panel that earns a regression
line. Panel (d) is the crux: total July rainfall against the number of wet days, with iso-intensity rays
for the normal (0.36 in per wet day) and for 2026 (0.89), which shows visually that 2026 sits far off the
normal intensity ray rather than simply high on the total axis. Panel (e) is the day-by-day trace of 2026
with normal high and low overlaid and daily precipitation on the right axis, marking the 2–4 July flooding
and the 27 July tornado day.


![July 2026 at Chicago O'Hare: mean temperature, precipitation, overnight minima, rainfall concentration and the day-by-day trace, with 1970-2026 rankings](03_climatological_context_outputs/july_climatology.png)

*Figure 1. July 2026 at Chicago O'Hare. Daily observations from the Iowa Environmental Mesonet,
1970–2026 (57 complete Julys). Anomalies and percentiles are against the 1991–2020 normal period.
Trend lines are drawn only where the least-squares trend is significant at p < 0.05 — of ten metrics
tested, only the overnight minimum temperature qualifies.*


The code that builds Figure 1 is given below in cleaned form. It is the reproduction path, not a re-run:
the daily dataframe `S` assembled in Section 1.2 and the `trends` dictionary from Section 1.5 are its only
inputs.

In [ ]:
apply_figure_style(sizes=(10, 9, 8))
fig1 = plt.figure(figsize=(15.0, 10.6))


def year_bars(ax, series, ylabel, title, panel, trend=None, norm_band=True):
    """Year-by-year bars with 2026 highlighted, the normal line, and an optional trend."""
    colours = ["#c0201a" if y == 2026 else "#b8c4d0" for y in series.index]
    ax.bar(series.index, series.values, color=colours, width=0.78, zorder=3)
    nm = series.loc[1991:2020].mean()
    ax.axhline(nm, color="#1a7a3d", ls="--", lw=1.4, zorder=5)
    if norm_band:
        lo, hi = np.percentile(series.loc[1991:2020], [25, 75])
        ax.axhspan(lo, hi, color="#1a7a3d", alpha=0.08, lw=0, zorder=1)
    # the restraint rule, in code: no line unless it is significant
    if trend is not None and trend.pvalue < 0.05:
        xs = np.array([series.index.min(), series.index.max()], dtype=float)
        ax.plot(xs, trend.intercept + trend.slope * xs, color="#1f4e8c", lw=1.8, zorder=6)
    ax.set_ylabel(ylabel)
    ax.set_xlim(series.index.min() - 1, series.index.max() + 1)
    ax.grid(axis="y", alpha=0.25, zorder=0)
    ax.set_title(f"({panel})  {title}", loc="left", fontsize=9.8, pad=6)
    return nm


# (a) mean temperature - no trend line, p = 0.079
axA = fig1.add_axes([0.055, 0.700, 0.415, 0.175])
nmT = year_bars(axA, S.tavg, "Mean temperature (\u00b0F)",
                "July mean temperature \u2014 13th warmest of 57", "a", trend=trends["tavg"])
axA.annotate(f"2026: {S.loc[2026, 'tavg']:.1f} \u00b0F\n+2.2 \u00b0F vs normal",
             xy=(2026, S.loc[2026, "tavg"]), xytext=(2008, 79.6),
             fontsize=7.6, color="#c0201a", fontweight="bold",
             arrowprops=dict(arrowstyle="->", lw=0.9, color="#c0201a"))
axA.text(1971, nmT - 1.9,
         f"1991\u20132020 normal {nmT:.1f} \u00b0F\n(shading = normal-period quartiles)",
         fontsize=6.9, color="#1a7a3d", fontweight="bold", va="top")
axA.set_ylim(70, 81)

# (b) total precipitation - also no trend line, p = 0.057
axB = fig1.add_axes([0.545, 0.700, 0.415, 0.175])
nmP = year_bars(axB, S.precip, "Total precipitation (in)",
                "July precipitation \u2014 9th wettest of 57", "b", trend=trends["precip"])
axB.annotate(f"2026: {S.loc[2026, 'precip']:.2f} in\n1.5\u00d7 normal",
             xy=(2026, S.loc[2026, "precip"]), xytext=(2006, 9.4),
             fontsize=7.6, color="#c0201a", fontweight="bold",
             arrowprops=dict(arrowstyle="->", lw=0.9, color="#c0201a"))
axB.text(1971, 10.6, f"1991\u20132020 normal {nmP:.2f} in",
         fontsize=6.9, color="#1a7a3d", fontweight="bold", va="top")
axB.set_ylim(0, 11.5)

# (c) overnight minimum - the ONLY panel that earns a regression line
axC = fig1.add_axes([0.055, 0.455, 0.415, 0.185])
year_bars(axC, S.tmin, "Mean daily minimum (\u00b0F)",
          "Overnight lows: the only significant trend, +0.48 \u00b0F/decade", "c",
          trend=trends["tmin"])
axC.text(1971, 71.4,
         f"trend +{trends['tmin'].slope * 10:.2f} \u00b0F per decade  "
         f"(p = {trends['tmin'].pvalue:.3f})",
         fontsize=7.2, color="#1f4e8c", fontweight="bold", va="top")
axC.set_ylim(60, 72.5)

# (d) total vs wet days, with iso-intensity rays
axD = fig1.add_axes([0.545, 0.455, 0.415, 0.185])
e26 = CC.loc[2026]
axD.scatter(CC.wet_days, CC.total, s=34,
            c=["#c0201a" if y == 2026 else "#8fa8bf" for y in CC.index],
            ec="white", lw=0.5, zorder=3)
axD.scatter([e26.wet_days], [e26.total], s=150, facecolor="none",
            ec="#c0201a", lw=2.0, zorder=4)
axD.annotate("2026\n5.36 in on just 6 wet days", xy=(e26.wet_days, e26.total),
             xytext=(11.5, 8.6), fontsize=7.6, color="#c0201a", fontweight="bold",
             arrowprops=dict(arrowstyle="->", lw=0.9, color="#c0201a"))
xs = np.linspace(2, 22, 30)
axD.plot(xs, 0.36 * xs, ls=":", lw=1.2, color="#4a4a4a", zorder=2)
axD.text(21.5, 0.36 * 21.5, "normal intensity\n0.36 in/wet day",
         fontsize=6.6, ha="right", va="bottom", color="#4a4a4a")
axD.plot(xs, 0.89 * xs, ls=":", lw=1.2, color="#c0201a", zorder=2)
axD.text(0.97, 0.62, "2026: 0.89 in per wet day", transform=axD.transAxes,
         fontsize=6.6, color="#c0201a", ha="right", va="bottom", fontweight="bold")
axD.set_xlabel("Number of wet days (\u22650.01 in)")
axD.set_ylabel("July total (in)")
axD.set_xlim(2, 22)
axD.set_ylim(0, 11.5)
axD.grid(alpha=0.25, zorder=0)
axD.set_title("(d)  A wet month delivered in very few events", loc="left", fontsize=9.8, pad=6)

# (e) day-by-day trace for July 2026
axE = fig1.add_axes([0.055, 0.180, 0.905, 0.185])
d26 = J[J.yr == 2026].sort_values("day").reset_index(drop=True)
dom = d26.day.dt.day.values
axE.fill_between(dom, d26.min_temp_f, d26.max_temp_f, color="#e0a080", alpha=0.55,
                 zorder=3, label="daily range")
axE.plot(dom, d26.max_temp_f, color="#b3202a", lw=1.5, zorder=4, label="daily max")
axE.plot(dom, d26.min_temp_f, color="#1f4e8c", lw=1.5, zorder=4, label="daily min")
axE.plot(dom, d26.climo_high_f, color="0.35", lw=1.1, ls="--", zorder=5, label="normal high/low")
axE.plot(dom, d26.climo_low_f, color="0.35", lw=1.1, ls="--", zorder=5)
axE.axhline(90, color="#8b1a1a", lw=0.9, ls=":", zorder=2)
axE.text(0.6, 90.7, "90 \u00b0F", fontsize=6.8, color="#8b1a1a")
axE.set_ylabel("Temperature (\u00b0F)")
axE.set_xlabel("Day of July 2026")
axE.set_xlim(0.5, 31.5)
axE.set_ylim(55, 110)
axE.set_xticks(range(1, 32, 2))
axE.grid(alpha=0.22, zorder=0)

axE2 = axE.twinx()
axE2.bar(dom, d26.precip_in, color="#1a5c8a", width=0.62, zorder=6, alpha=0.85)
axE2.set_ylabel("Precipitation (in)", color="#1a5c8a")
axE2.tick_params(axis="y", labelcolor="#1a5c8a")
axE2.set_ylim(0, 6.0)
for x_, v in zip(dom, d26.precip_in):
    if v >= 1.0:
        axE2.text(x_, v + 0.12, f"{v:.2f}", ha="center", fontsize=6.9,
                  color="#1a5c8a", fontweight="bold")
for x_, lab in [(3, "2\u20134 Jul\nflooding"), (27, "27 Jul\ntornadoes")]:
    axE.annotate(lab, xy=(x_, 97), xytext=(x_, 103.5), fontsize=7.0, color="#8b1a1a",
                 ha="center", fontweight="bold",
                 arrowprops=dict(arrowstyle="-", lw=0.8, color="#8b1a1a"))
axE.legend(loc="lower left", fontsize=7.0, ncol=4, framealpha=0.93)
axE.set_title("(e)  Day by day: three rain events, and a hot, humid second half",
              loc="left", fontsize=9.8, pad=6)

# (f) summary strip - the headline table as figure text
axF = fig1.add_axes([0.055, 0.030, 0.905, 0.105])
axF.axis("off")
summary = [("Mean temperature", "76.8 \u00b0F", "+2.2", "13 / 57", "79th"),
           ("Mean overnight low", "68.0 \u00b0F", "+2.6", "10 / 57", "84th"),
           ("Days \u2265 95 \u00b0F", "4", "+3", "8 / 57", "88th"),
           ("Mean max dewpoint", "68.4 \u00b0F", "+1.8", "15 / 57", "75th"),
           ("Total precipitation", "5.36 in", "+1.77", "9 / 57", "86th"),
           ("Wet days", "6", "\u22123.8", "55 / 57", "9th"),
           ("Days \u2265 1.00 in", "3", "+2.2", "5 / 57", "100th"),
           ("Rain per wet day", "0.89 in", "+0.53", "2 / 57", "98th")]
for (lab, val, anom, rk, pc), x0 in zip(summary, np.linspace(0.005, 0.88, len(summary))):
    col = "#c0201a" if pc in ("100th", "98th", "9th", "88th") else "0.2"
    axF.text(x0, 0.86, lab, fontsize=7.3, fontweight="bold", color="0.15", transform=axF.transAxes)
    axF.text(x0, 0.58, val, fontsize=9.4, fontweight="bold", color=col, transform=axF.transAxes)
    axF.text(x0, 0.32, f"{anom} vs normal", fontsize=6.8, color="0.4", transform=axF.transAxes)
    axF.text(x0, 0.08, f"rank {rk} \u00b7 {pc}", fontsize=6.8, color=col, transform=axF.transAxes)

fig1.suptitle("July 2026 at Chicago O'Hare \u2014 how hot, how wet, and how unusual",
              x=0.055, ha="left", fontsize=13.5, y=0.985)
fig1.text(0.055, 0.950,
          "Daily observations from the Iowa Environmental Mesonet, 1970\u20132026 (57 complete "
          "Julys). Anomalies and percentiles are against the 1991\u20132020 normal period.\n"
          "Trend lines are drawn only where the least-squares trend is significant at p < 0.05 "
          "\u2014 of ten metrics tested, only the overnight minimum temperature qualifies.",
          fontsize=8.4, color=META_GREY, ha="left", va="top")

fig1.savefig("july_climatology.png", dpi=200, bbox_inches="tight")


### 1.7 Figure 2 — the same season from four other angles

Figure 2 was built earlier in the analysis over a shorter window (1990–2026) and shows four
complementary views: July totals by year, the distribution of *daily* maximum dewpoint across all July
days with 27 July 2026 marked, the count of 1-inch rain days by year, and the cumulative July rainfall
trace for 2026 against the 1991–2020 median and 10th–90th percentile envelope.

The dewpoint panel is the one worth quoting on air. The 27 July daily maximum dewpoint at O'Hare was
**79 °F, the 98.8th percentile of all July days in the 1990–2026 record** — a value that occurs about
0.4 times per July. That is the surface signature of the same tropical column the sounding analysis
found aloft.

> **Watch the denominator.** Figure 2 uses 1990–2026 (37 complete Julys) while Figure 1 uses 1970–2026
> (57). July 2026 therefore appears as "7th wettest of 37" in Figure 2 and "9th wettest of 57" in
> Figure 1. Both are correct; they are different reference sets. This is exactly the kind of discrepancy
> that reads as an error in a published figure pair, so the window belongs in every caption. Where a
> single number is quoted in this notebook's prose, it is the 57-year one.


![Four-panel seasonal context for Chicago O'Hare 1990-2026: July precipitation by year, the distribution of daily maximum dewpoint, counts of one-inch rain days, and the cumulative July rainfall trace for 2026 against the normal envelope](03_climatological_context_outputs/seasonal_context.png)

*Figure 2. Placing the events in a normal season — Chicago O'Hare, 1990–2026. Source: NWS/ASOS daily
climate record via the Iowa Environmental Mesonet. Panel (d) shows that two events did nearly all the
work: the cumulative trace tracks the normal median until 2 July, jumps, flattens for three weeks, then
jumps again on the 27th.*

In [ ]:
apply_figure_style(sizes=(10, 9, 8))
fig2, axes2 = plt.subplots(2, 2, figsize=(14.5, 8.4))
fig2.subplots_adjust(hspace=0.36, wspace=0.24)

# Restrict to the 1990-2026 window used for this figure, and say so in the caption.
cl = daily[daily.yr >= 1990].copy()
julc = cl[cl.mo == 7]

Jp = julc.groupby("yr")["precip_in"].agg(["sum", "count"])
Jp = Jp[Jp["count"] >= 28]["sum"]
norm_p = Jp.loc[1991:2020].mean()

# (a) July totals by year
ax = axes2[0, 0]
ax.bar(Jp.index, Jp.values, color=["#b03060" if y == 2026 else "#b8c4d0" for y in Jp.index], zorder=3)
ax.axhline(norm_p, color="#1a7a3d", lw=1.3, ls="--", zorder=4)
ax.text(Jp.index.min() + 0.4, norm_p + 0.28, f"1991\u20132020 normal, {norm_p:.2f} in",
        fontsize=7.8, color="#1a7a3d")
ax.annotate(f"2026: {Jp.loc[2026]:.2f} in\n7th wettest of {len(Jp)}",
            xy=(2026, Jp.loc[2026]), xytext=(2018.5, 8.3), fontsize=8, color="#b03060",
            arrowprops=dict(arrowstyle="->", lw=0.9, color="#b03060"))
ax.set_ylabel("July precipitation (in)")
ax.set_xlabel("Year")
ax.set_title("July 2026 was 1.5\u00d7 the normal monthly rainfall", loc="left", pad=6)
panel_letter(ax, "a")

# (b) distribution of daily maximum dewpoint across all July days
ax = axes2[0, 1]
dwp = julc.max_dewpoint_f.dropna()
ax.hist(dwp, bins=np.arange(40, 86, 2), color="#b8c4d0", zorder=3)
ax.axvline(79, color="#b03060", lw=1.8, zorder=5)
ax.axvline(dwp.mean(), color="#1a7a3d", lw=1.3, ls="--", zorder=5)
ax.text(dwp.mean() - 0.8, ax.get_ylim()[1] * 0.90, f"July mean\n{dwp.mean():.0f} \u00b0F",
        fontsize=7.8, color="#1a7a3d", ha="right")
ax.text(78.2, ax.get_ylim()[1] * 0.62,
        f"27 Jul 2026: 79 \u00b0F\n{(dwp <= 79).mean() * 100:.1f}th percentile\n"
        "(0.4 such days per July)",
        fontsize=7.8, color="#b03060", ha="right")
ax.set_xlabel("Daily maximum dewpoint (\u00b0F)")
ax.set_ylabel("Number of July days")
ax.set_title("The tornado day's moisture was near-record for July", loc="left", pad=6)
panel_letter(ax, "b")

# (c) count of 1-inch rain days by year
ax = axes2[1, 0]
hv = julc.groupby("yr")["precip_in"].apply(lambda s: int((s >= 1.0).sum()))
ax.bar(hv.index, hv.values, color=["#b03060" if y == 2026 else "#b8c4d0" for y in hv.index], zorder=3)
nm_hv = hv.loc[1991:2020].mean()
ax.axhline(nm_hv, color="#1a7a3d", lw=1.3, ls="--", zorder=4)
ax.text(hv.index.min() + 0.4, nm_hv + 0.12, f"1991\u20132020 normal, {nm_hv:.1f} days",
        fontsize=7.8, color="#1a7a3d")
ax.set_ylabel("July days with \u22651.00 in")
ax.set_xlabel("Year")
ax.set_title("Three 1-inch rain days in July 2026, against a normal of one", loc="left", pad=6)
panel_letter(ax, "c")

# (d) cumulative trace, 2026 against the normal-period envelope
ax = axes2[1, 1]
jl = julc.copy()
jl["dom"] = jl.day.dt.day
piv = jl.pivot_table(index="dom", columns="yr", values="precip_in").fillna(0).cumsum()
hist = piv[[y for y in piv.columns if 1991 <= y <= 2020]]
ax.fill_between(piv.index, hist.quantile(0.1, axis=1), hist.quantile(0.9, axis=1),
                color="#b8c4d0", alpha=0.6, zorder=2, label="1991\u20132020, 10th\u201390th percentile")
ax.plot(piv.index, hist.median(axis=1), color="#1a7a3d", lw=1.4, ls="--", zorder=3,
        label="1991\u20132020 median")
ax.plot(piv.index, piv[2026], color="#b03060", lw=2.0, zorder=4, label="2026")
ax.set_ylim(-0.2, 7.4)
ax.annotate("2\u20134 Jul\nflooding", xy=(4, piv.loc[4, 2026]), xytext=(7.5, 1.6),
            fontsize=7.6, color="#b03060",
            arrowprops=dict(arrowstyle="->", lw=0.8, color="#b03060"))
ax.annotate("27 Jul\ntornadoes", xy=(27, piv.loc[27, 2026]), xytext=(19.0, 6.6),
            fontsize=7.6, color="#b03060",
            arrowprops=dict(arrowstyle="->", lw=0.8, color="#b03060"))
ax.set_xlabel("Day of July")
ax.set_ylabel("Cumulative July precipitation (in)")
ax.set_xlim(1, 31)
ax.legend(loc="upper left", fontsize=7.8, bbox_to_anchor=(0.02, 0.80))
ax.set_title("Two big events did most of the work", loc="left", pad=6)
panel_letter(ax, "d")

fig2.suptitle("Placing the events in a normal season \u2014 Chicago O'Hare, 1990\u20132026",
              x=0.075, ha="left", fontsize=12.5, y=0.98)
fig2.text(0.075, 0.938,
          "Source: NWS/ASOS daily climate record via the Iowa Environmental Mesonet.",
          fontsize=8.5, color=META_GREY, ha="left")

fig2.savefig("seasonal_context.png", dpi=200, bbox_inches="tight")


## 2. A purpose-built precipitable-water climatology

### 2.1 Why build one at all

Notebook 2 established that the 27 July inflow column held about 2.3 inches of precipitable water
(PWAT) over the tornado corridor. On its own that number means nothing to a listener, and — worse — it
is easy to abuse. Quoting a model's percentile against the model's own history compounds any model bias,
and the reanalysis-based PWAT climatologies in common use are coarse in both space and time.

So the question was posed differently: **how does this column compare to every July column actually
measured by a radiosonde near Chicago?** Chicago itself has no upper-air site. The three that bracket the
region's inflow are:

| Site | Location | Role |
|---|---|---|
| **KILX** | Lincoln, Illinois | Directly upstream in southwesterly flow — the nearest representative sounding for Chicagoland inflow |
| **KDVN** | Davenport / Quad Cities, Iowa | Upstream to the west |
| **KGRB** | Green Bay, Wisconsin | Downstream/north, samples the air mass north of the warm front |

Every July sounding at these three sites from 2000 through 2025 was retrieved and its PWAT computed by
integrating the **observed** dewpoint profile from the surface to 300 hPa. That gives **5,025 soundings**
— a distribution dense enough to quote 99th percentiles with some confidence.

### 2.2 The integration

PWAT is the depth of liquid water that would result from condensing the entire vapour column:

$$\mathrm{PWAT} = \frac{1}{\rho_w g}\int_{p_{sfc}}^{p_{top}} q \, dp$$

with specific humidity $q$ derived from the observed dewpoint via saturation vapour pressure at the
dewpoint (Bolton 1980), the mixing ratio $w = 0.622\,e/(p-e)$, and $q = w/(1+w)$. The integration is
truncated at 300 hPa, above which the contribution is negligible and reported moisture is unreliable.

Quality control is deliberately blunt and applied identically to every sounding: at least five valid
levels, and a lowest reported level at or below 800 hPa so that a sounding truncated near the surface
cannot masquerade as a dry column.

In [ ]:
G = 9.80665       # m s-2
RHO_W = 1000.0    # kg m-3


def esat_hpa(tc):
    """Saturation vapour pressure over water, Bolton (1980), hPa."""
    return 6.112 * np.exp(17.67 * tc / (tc + 243.5))


def pwat_mm(p_hpa, tc, tdc, ptop=300.0):
    """Precipitable water (mm) by integrating specific humidity over pressure.

    Evaluated from the OBSERVED dewpoint profile. Returns NaN if the sounding
    fails QC: fewer than five valid levels, or no level at or below 800 hPa.
    """
    m = np.isfinite(p_hpa) & np.isfinite(tc) & np.isfinite(tdc)
    p, td = p_hpa[m], tdc[m]
    if p.size < 5:
        return np.nan
    order = np.argsort(-p)              # surface first
    p, td = p[order], td[order]
    keep = p >= ptop
    p, td = p[keep], td[keep]
    if p.size < 5 or p[0] < 800:        # require a genuine surface-based profile
        return np.nan
    e = esat_hpa(td)                    # actual vapour pressure from dewpoint
    w = 0.622 * e / (p - e)             # mixing ratio, kg/kg
    q = w / (1.0 + w)                   # specific humidity
    # p converted to Pa; the array descends in pressure so negate the integral
    return float(-np.trapezoid(q, p * 100.0) / (RHO_W * G) * 1000.0)


### 2.3 Validating against MetPy

A hand-rolled integration is exactly the kind of code that produces a plausible-looking but wrong
number, and a whole climatology built on it would be wrong consistently — the most dangerous failure
mode, because internal consistency hides it. So it is checked against
`metpy.calc.precipitable_water` on the same sounding before being trusted.

The two agree to **within 0.6 mm on a ~49 mm column (48.8 mm here versus 49.4 mm from MetPy, about 1.2
percent)**. The residual is expected and benign: MetPy integrates in mixing-ratio form with its own
vapour-pressure formulation and its own handling of the top boundary, and the 800 hPa QC gate used here
is not part of MetPy's contract. A percent-level offset is far smaller than the spread of the
climatological distribution, so percentile placement is unaffected.

In [ ]:
import metpy.calc as mpcalc
from metpy.units import units

# One sounding, fetched as a single-profile check case: KILX at 12 UTC on 27 July 2026.
u_one = (f"{IEM}/raob.py?station=KILX"
         "&sts=2026-07-27T12:00Z&ets=2026-07-27T13:00Z&fmt=csv")
one = pd.read_csv(io.StringIO(requests.get(u_one, timeout=120).text), na_values=["M", ""])
one = one.dropna(subset=["pressure_mb", "tmpc", "dwpc"]).sort_values("pressure_mb", ascending=False)

ours = pwat_mm(one.pressure_mb.values, one.tmpc.values, one.dwpc.values)

sel = one[one.pressure_mb >= 300]
metpy_pwat = mpcalc.precipitable_water(
    sel.pressure_mb.values * units.hPa,
    sel.dwpc.values * units.degC,
    top=300 * units.hPa,
).to("mm").m

print(f"this notebook : {ours:.1f} mm  ({ours / 25.4:.2f} in)")
print(f"MetPy         : {metpy_pwat:.1f} mm  ({metpy_pwat / 25.4:.2f} in)")
print(f"difference    : {abs(ours - metpy_pwat):.1f} mm "
      f"({abs(ours - metpy_pwat) / metpy_pwat * 100:.1f} %)")


### 2.4 Fetching 27 years of soundings

This is where the IEM bulk radiosonde endpoint earns its keep, and where the gotchas live.

> **Gotcha — use IEM, not the University of Wyoming CGI.** The long-standing
> `weather.uwyo.edu/cgi-bin/sounding` endpoint has **moved and now returns 404**. Any notebook or script
> that reaches for it will fail. The IEM `raob.py` service is a drop-in replacement with a better bulk
> interface.
>
> **Gotcha — one request per site-month, not per sounding.** `raob.py?station=KILX&sts=...&ets=...&fmt=csv`
> returns **every sounding in the time window, one row per reported level**. A whole July at one site is
> a single request. A 27-year, 3-station climatology is therefore about **81 requests** and completes in a
> few minutes. Requesting soundings individually would be ~5,000 requests, would take hours, and would
> be discourteous to a free service.
>
> **Gotcha — group by `validUTC`.** Because the response is level-major, consecutive soundings are only
> separable by their valid time. Group on `validUTC` before integrating, or the 12Z and 00Z profiles get
> concatenated into one nonsensical column.
>
> **Gotcha — the end time must be *strictly* later than the start time.** `ets` equal to `sts` returns an
> empty body with HTTP 200 — no error, no rows, and a silently empty climatology. When checking a single
> sounding, ask for a one-hour window as in the validation cell above.

The retrieval loop below is `fetch_pwat_clim.py` from the companion scripts. It is reproduced for
completeness and **is not re-run in this notebook**; the cached `pwat_climatology_july.csv` is read
instead.

In [ ]:
STATIONS = ["KILX", "KDVN", "KGRB"]   # Lincoln IL (upstream), Quad Cities, Green Bay
YEARS = range(2000, 2027)

# --- fetch_pwat_clim.py, reproduced. Do not re-run: ~81 requests, several minutes. ---
def build_pwat_climatology(stations=STATIONS, years=YEARS, out="pwat_climatology_july.csv"):
    rows = []
    for stn in stations:
        for yr in years:
            # NOTE: ets is 1 August, strictly later than sts - an equal ets returns nothing.
            u = (f"{IEM}/raob.py?station={stn}"
                 f"&sts={yr}-07-01T00:00Z&ets={yr}-08-01T00:00Z&fmt=csv")
            d = None
            for attempt in range(3):
                try:
                    d = pd.read_csv(io.StringIO(requests.get(u, timeout=300).text),
                                    na_values=["M", ""])
                    break
                except Exception as exc:
                    if attempt == 2:
                        print(f"  {stn} {yr} FAIL {type(exc).__name__}", flush=True)
                    time.sleep(5)
            if d is None or not len(d) or "validUTC" not in d.columns:
                continue
            d["validUTC"] = pd.to_datetime(d.validUTC)
            n = 0
            # one sounding per validUTC group; level-major response demands this
            for t, g in d.groupby("validUTC"):
                pw = pwat_mm(g.pressure_mb.values, g.tmpc.values, g.dwpc.values)
                if np.isfinite(pw):
                    rows.append(dict(station=stn, valid=t, pwat_mm=pw))
                    n += 1
            print(f"{stn} {yr}: {n} soundings", flush=True)
    PW = pd.DataFrame(rows)
    PW.to_csv(out, index=False)
    return PW


# Read the cached product rather than re-fetching.
PW = pd.read_csv("pwat_climatology_july.csv")
PW["valid"] = pd.to_datetime(PW.valid)
PW["yr"] = PW.valid.dt.year

print(f"{len(PW):,} July soundings")
print(PW.groupby("station").pwat_mm.describe().round(1).to_string())


### 2.5 Placing the event on the distribution

For each site, the 27 July 2026 12 UTC sounding is compared to that site's own 2000–2025 July
distribution (2026 itself excluded from the reference set, so the event is not scored against itself).
The percentile is the same weak percentile rank defined in Section 1.3.

In [ ]:
event = pd.Timestamp("2026-07-27 12:00")

res = []
for stn, g in PW.groupby("station"):
    clim = g[g.yr < 2026].pwat_mm          # 2026 excluded from its own reference set
    row = g[g.valid == event]
    if not len(row):
        print(stn, "no 27 Jul 12Z sounding")
        continue
    obs = float(row.pwat_mm.iloc[0])
    res.append(dict(station=stn, n_clim=len(clim),
                    obs_mm=round(obs, 1), obs_in=round(obs / 25.4, 2),
                    median_in=round(clim.median() / 25.4, 2),
                    p90_in=round(np.percentile(clim, 90) / 25.4, 2),
                    p99_in=round(np.percentile(clim, 99) / 25.4, 2),
                    max_in=round(clim.max() / 25.4, 2),
                    percentile=round((clim <= obs).mean() * 100, 1)))
PCT = pd.DataFrame(res)
print(PCT.to_string(index=False))

# KILX is the reference distribution used for the model field, being the nearest upstream site.
clim_ilx = PW[(PW.station == "KILX") & (PW.yr < 2026)].pwat_mm
print(f"\nKILX July median: {clim_ilx.median() / 25.4:.2f} in  "
      f"(n = {len(clim_ilx):,})")


### 2.6 Results, and an honest framing

**Observed soundings, 27 July 2026 12 UTC:**

| Site | PWAT | Percentile of July climatology |
|---|---|---|
| KDVN (Quad Cities) | 1.66 in | 80th |
| KILX (Lincoln IL) | 1.92 in | 90th |
| KGRB (Green Bay) | 2.13 in | **99.4th** |
| KILX at 28 Jul 00 UTC | 2.09 in | 97th |

The KILX column *rose* through the event, from the 90th percentile at 12 UTC to the 97th by 00 UTC on the
28th — moisture was still being advected in behind the storms. Green Bay at the 99.4th percentile shows
how far north the tropical air mass had pushed.

**HRRR 17 UTC field, scored against the KILX distribution:**

| Quantity | PWAT | Percentile |
|---|---|---|
| Domain mean | 1.83 in | 84th |
| Tornado corridor | 2.29 in | **99.4th** |
| Domain 99th percentile | 2.45 in | — |

The July KILX median is **1.38 inches**, so the corridor value of 2.29 inches was **1.7 times a normal
July column**.

> **The honest framing.** Areawide, this was a top 10–20 percent moisture day — genuinely moist, not
> unprecedented. **Locally, along the corridor that produced the tornadoes, it was a 99th-percentile
> column.** Only 25 percent of the model domain exceeded 2.00 inches and only 3 percent exceeded 2.25
> inches. The moisture anomaly was *narrow*.
>
> That narrowness is the point. **The rarity of 27 July was not the moisture alone — it was the
> combination of a 99th-percentile local column with anomalous deep-layer shear.** High PWAT days are
> common in a Chicago July; 65 kt of 0–6 km shear on such a day is not. Attributing the outbreak to
> moisture alone would be wrong, and would also be the easiest thing for a listener to have seen before
> without a tornado following.

> **Correction to an earlier framing.** During the analysis the corridor value was described as "nearly
> double" a normal July column. That overstates it. Against the KILX July median of 1.38 inches,
> 2.29 inches is **1.7 times**, not 2 times. The corrected multiplier is used throughout this notebook.
> The distinction matters less for the physics than for the habit: an anomaly quoted as a ratio needs
> its denominator named, and 1.7× against a stated median is a defensible statement in a way that
> "nearly double" is not.

> **Caution — never quote a model's domain maximum as an environmental value.** The raw HRRR PWAT maximum
> over the event window reaches **3.50 inches**, which would be near-record for this latitude and is a
> tempting headline. It is not an environmental quantity. Only **104 of 659,984 pixels** exceed 3.0
> inches, and they sit inside deep convective cores where the integrated column includes **condensed**
> water — cloud and precipitation hydrometeors, not vapour available to be lifted. Use the domain **99th
> percentile (2.45 in)** or a specific value at an inflow-region point instead. A grid-point maximum over
> a convective field is a property of the convection, not of the air feeding it.


### 2.7 Figure 3 — moisture in percentile terms

Panel (a) maps the HRRR PWAT field at the tornado hour with the KILX climatological median, 90th and
99th percentiles overlaid as contours, so the reader can see directly which parts of the domain were
extreme rather than merely moist. Panel (b) is the full distribution of 5,025 July soundings at Lincoln
with the observed and model values marked. Panel (c) collects the percentile placements from Section 2.6
on one axis, distinguishing observed soundings (blue) from model values (orange and red).


![Precipitable water in percentile terms: the HRRR field at the tornado hour with climatological percentile contours, the distribution of 5025 July soundings at Lincoln Illinois, and percentile placements for each observed and model value](03_climatological_context_outputs/pwat_percentile.png)

*Figure 3. How unusual was the moisture? Climatology: 5,025 July radiosondes at Lincoln IL, Quad Cities
and Green Bay, 2000–2025, with precipitable water integrated from the observed moisture profile
(surface–300 hPa; validated to within 0.6 mm of MetPy). Event values are the 27 July 12 UTC soundings and
the HRRR analysis at 17 UTC. Note that the 99th-percentile contour in panel (a) encloses only a narrow
corridor.*

In [ ]:
import pickle

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.patheffects as pe
from matplotlib.lines import Line2D

# HRRR fields for the tornadic event, from fetch_hrrr.py (notebook 2).
D = pickle.load(open("hrrr_chicago_derived.pkl", "rb"))
M, LAT, LON = D["maps"], D["lat"], D["lon"]
VT = pd.DatetimeIndex(D["valid"])

it = list(VT).index(pd.Timestamp("2026-07-27 17:00"))
pw_in = M["pwat_in"][it]

PC = ccrs.PlateCarree()
apply_figure_style(sizes=(10, 9, 8))
fig3 = plt.figure(figsize=(14.6, 8.2))

# (a) PWAT field with climatological percentile contours
axA = fig3.add_axes([0.055, 0.235, 0.40, 0.46], projection=PC)
axA.set_extent([-92.3, -84.3, 39.3, 44.3], crs=PC)
axA.add_feature(cfeature.LAKES.with_scale("50m"), fc="#dceaf2", ec="0.2", lw=0.7, zorder=2)
axA.add_feature(cfeature.STATES.with_scale("50m"), ec="0.35", lw=0.7, zorder=3)
cf = axA.contourf(LON, LAT, pw_in, levels=np.arange(1.0, 2.81, 0.1),
                  cmap="BrBG", extend="both", transform=PC, zorder=1)
cax = fig3.add_axes([0.462, 0.235, 0.012, 0.46])
cb = fig3.colorbar(cf, cax=cax)
cb.set_label("Precipitable water (in)", fontsize=8.5)
cb.ax.tick_params(labelsize=7.5)

# climatological thresholds from the KILX distribution, drawn ON the model field
p50, p90, p99 = [np.percentile(clim_ilx, q) / 25.4 for q in (50, 90, 99)]
cs = axA.contour(LON, LAT, pw_in, levels=[p50, p90, p99],
                 colors=["0.35", "#b8860b", "#8b1a1a"], linewidths=[1.0, 1.5, 2.0],
                 transform=PC, zorder=5)
axA.clabel(cs, fmt={p50: "median", p90: "90th", p99: "99th"}, fontsize=6.8, inline=True)

axA.plot(-88.08, 41.88, marker="*", ms=16, mfc="k", mec="white", mew=1.2, transform=PC, zorder=8)
for nm, la, lo in [("Chicago", 41.878, -87.630), ("Lincoln IL\n(KILX)", 40.15, -89.34),
                   ("Quad Cities\n(KDVN)", 41.61, -90.58)]:
    axA.plot(lo, la, "o", ms=3.4, color="k", transform=PC, zorder=8)
    axA.text(lo + 0.09, la + 0.07, nm, fontsize=6.8, transform=PC, zorder=8,
             path_effects=[pe.withStroke(linewidth=2.0, foreground="white")])
axA.set_title("(a)  Precipitable water at the tornado hour (17 UTC)\n"
              "contours = July climatological percentiles", loc="left", fontsize=9.6, pad=6)

# (b) the observed distribution with the event marked
axB = fig3.add_axes([0.560, 0.565, 0.405, 0.275])
axB.hist(clim_ilx / 25.4, bins=np.arange(0.3, 2.75, 0.06),
         color="#b8c4d0", ec="white", lw=0.4, zorder=3)
axB.axvline(1.92, color="#1f4e8c", lw=2.0, zorder=5)
axB.text(1.88, 78, "KILX observed 12Z\n1.92 in \u2014 90th pctile",
         fontsize=7.2, color="#1f4e8c", fontweight="bold", va="top", ha="right")
axB.axvline(2.29, color="#c0201a", lw=2.0, zorder=5)
axB.text(2.33, 78, "HRRR Glen Ellyn 17Z\n2.29 in \u2014 99.4th pctile",
         fontsize=7.2, color="#c0201a", fontweight="bold", va="top", ha="left")
axB.set_xlabel("Precipitable water (in)")
axB.set_ylabel("Number of soundings")
axB.set_title(f"(b)  All {len(clim_ilx):,} July soundings at Lincoln IL, 2000\u20132025",
              loc="left", fontsize=9.6, pad=6)
axB.grid(axis="y", alpha=0.25, zorder=0)

# (c) percentile placements, observed versus model
axC = fig3.add_axes([0.560, 0.10, 0.405, 0.305])
labels = ["KDVN\n12Z", "KILX\n12Z", "KGRB\n12Z", "KILX\n00Z (28th)",
          "HRRR\ndomain mean", "HRRR\nGlen Ellyn"]
pctl = [80.3, 90.2, 99.4, 97.1, 84.5, 99.4]
inches = [1.66, 1.92, 2.13, 2.09, 1.83, 2.29]
colours = ["#8fa8bf"] * 4 + ["#d99a3a", "#c0201a"]
x = np.arange(len(pctl))
axC.bar(x, pctl, color=colours, zorder=3, width=0.66)
for xi, v, q in zip(x, pctl, inches):
    axC.text(xi, v + 1.6, f"{v:.0f}th", ha="center", fontsize=7.8,
             fontweight="bold", color="0.2")
    axC.text(xi, 4, f"{q:.2f} in", ha="center", fontsize=7.0,
             color="white", fontweight="bold")
axC.axhline(90, color="#b8860b", ls="--", lw=1.2, zorder=4)
axC.axhline(99, color="#8b1a1a", ls="--", lw=1.2, zorder=4)
axC.set_xticks(x)
axC.set_xticklabels(labels, fontsize=7.2)
axC.set_ylabel("Percentile of July climatology")
axC.set_ylim(0, 118)
axC.grid(axis="y", alpha=0.25, zorder=0)
axC.legend(handles=[Line2D([], [], color="#b8860b", ls="--", lw=1.2, label="90th percentile"),
                    Line2D([], [], color="#8b1a1a", ls="--", lw=1.2, label="99th percentile")],
           loc="upper left", bbox_to_anchor=(0.0, 1.0), fontsize=7.0, frameon=False, ncol=2)
axC.set_title("(c)  Observed soundings (blue) and model field (orange/red)",
              loc="left", fontsize=9.6, pad=6)

fig3.suptitle("How unusual was the moisture? Precipitable water in percentile terms",
              x=0.055, ha="left", fontsize=13, y=0.975)
fig3.text(0.055, 0.925,
          "Climatology: 5,025 July radiosondes at Lincoln IL, Quad Cities and Green Bay, "
          "2000\u20132025, with precipitable water integrated from the observed moisture profile\n"
          "(surface\u2013300 hPa; validated to within 0.6 mm of MetPy). Event values are the "
          "27 Jul 12 UTC soundings and the HRRR analysis at 17 UTC.",
          fontsize=8.3, color=META_GREY, ha="left", va="top")

fig3.savefig("pwat_percentile.png", dpi=200, bbox_inches="tight")


## 3. The 31 July – 1 August flooding: a different storm entirely

### 3.1 The regime contrast

Four days after the tornadoes, the same metropolitan area was under flood warnings. The temptation is to
treat this as a continuation — more of the same tropical air mass doing more damage. The environment says
otherwise.

At Glen Ellyn, in the middle of both events:

| Quantity | 27 July (tornadic) | 1 August (flooding) |
|---|---|---|
| MLCAPE | 4350 J kg⁻¹ | under 350 J kg⁻¹, all day |
| 0–6 km bulk shear | 65 kt | ~15 kt |
| PWAT | 2.29 in | ~1.9 in |
| 2 m dewpoint | 79.6 °F | 64.7 °F |

**The moisture stayed; the dynamics collapsed.** MLCAPE fell by more than an order of magnitude and
deep-layer shear by a factor of four. That combination — a deep moist column with almost no instability
and almost no shear — is the signature of **efficient warm-rain and stratiform processes**, not deep
convection. Precipitation efficiency in such a column is high because the cloud-base-to-freezing-level
depth is large and warm and the entrainment of dry air is minimal, so nearly all condensate reaches the
ground. Rain rates are modest compared to a supercell's core, but they persist for many hours over the
same ground.

The practical consequence is that these two events required entirely different forecasts, different
warnings and different public messaging, despite drawing on the same air mass. That is a useful thing to
be able to say on air: *the same humidity that fed the tornado outbreak produced flooding four days later
by a completely different mechanism.*

### 3.2 The rainfall distribution, and where the model missed

Point rainfall reports across the metro ran **4 to 5.6 inches**, while the airport gauges — ORD, MDW, DPA,
PWK — recorded only **1 to 2.3 inches**. That is not an inconsistency; it is the signature of **narrow
training bands**, convective lines that repeatedly regenerate over the same corridor while a few
kilometres away almost nothing falls. HRRR captured the general area of heavy rain but **under-forecast
the local maxima**, which is the expected behaviour: a 3 km model with parameterised microphysics
smooths precisely the sub-grid banding that produces the peak totals.

> **Gotcha — do not sum `p01i` from the IEM ASOS service.** The `p01i` field is a **running hourly
> accumulation, repeated at every observation within the hour**. Summing the column double-counts
> severely: doing so produced a spurious **13.90 inches** at Midway, roughly six times the true value.
> Take the **hourly maximum** instead:
>
> ```python
> hourly = obs.dropna(subset=["p01i"]).set_index("valid")["p01i"].resample("1h").max()
> ```
>
> The failure is quiet — a plausible-looking large number with no error raised — which makes it exactly
> the sort of thing that reaches a broadcast before anyone checks it.

In [ ]:
def asos_obs(station, start, end):
    """Sub-hourly ASOS observations from IEM. NOTE: p01i is a RUNNING hourly total."""
    u = (f"{IEM}/asos.py?station={station}"
         "&data=tmpf&data=dwpf&data=sknt&data=gust&data=mslp&data=p01i&data=drct"
         f"&year1={start:%Y}&month1={start:%m}&day1={start:%d}&hour1={start:%H}&minute1=0"
         f"&year2={end:%Y}&month2={end:%m}&day2={end:%d}&hour2={end:%H}&minute2=0"
         "&tz=UTC&format=onlycomma&latlon=no&missing=M&trace=0.0001"
         "&direct=no&report_type=3&report_type=4")
    d = pd.read_csv(io.StringIO(requests.get(u, timeout=120).text), na_values=["M", "T"])
    d["valid"] = pd.to_datetime(d.valid, utc=True)
    return d


FLOOD_START = pd.Timestamp("2026-07-30 00:00")
FLOOD_END = pd.Timestamp("2026-08-02 00:00")

OBS = {}
for st in ["ORD", "MDW", "DPA", "PWK", "ARR", "GYY", "RFD", "JOT"]:
    for attempt in range(3):
        try:
            d = asos_obs(st, FLOOD_START, FLOOD_END)
            if "valid" not in d.columns:
                raise ValueError("unexpected columns")
            OBS[st] = d
            break
        except Exception as exc:
            if attempt == 2:
                print(st, "FAIL", type(exc).__name__)
            time.sleep(4)

# CORRECT: hourly maximum of the running accumulation.
# WRONG:   OBS[st].p01i.sum()  -> gave 13.90 in at MDW, ~6x reality.
event_totals = {}
for st, d in OBS.items():
    hourly = d.dropna(subset=["p01i"]).set_index("valid")["p01i"].resample("1h").max()
    event_totals[st] = round(float(hourly.sum()), 2)
print("Event storm totals from hourly maxima (in):", event_totals)


### 3.3 The key finding: a second crest on ground that never dried

This is the load-bearing result of the flooding analysis, and it is not about rainfall rate at all.

The river gauges that went under flood warning on 1 August **had already flooded on 27 July**. Over the
four intervening days they receded — but **not back to their pre-27 July baseline**. Soil moisture stayed
elevated, storage in the small urban watersheds of DuPage and southern Cook was not recovered, and
channels were still running high. When the 31 July – 1 August rain arrived, it fell on catchments that
were already primed, and the gauges **exceeded their 27 July crests**.

> **The 31 July – 1 August flooding was a second crest on ground that never dried out. The
> load-bearing explanation is antecedent wetness, not rainfall rate.**

This is why the July concentration statistic from Section 1.4 matters so much. Three 1-inch-plus days in
a single month, with 95 percent of the monthly total in three calendar days, is exactly the pattern that
produces sequential flooding: each pulse arrives before the catchment has recovered from the last. A
month with the same total spread over ten days would very likely have produced neither flood.

Four gauges were under NWS flood warning by the evening of 1 August:

| Gauge | River and location | Flood stage | Forecast crest | Observed |
|---|---|---|---|---|
| **THNI2** | Thorn Creek at Thornton | 9.0 ft | 10.0 ft | **11.31 ft, still rising — past its forecast crest** |
| **BOLI2** | East Branch DuPage River at Bolingbrook | 20.0 ft | 20.3 ft | above flood stage |
| **LCMI3** | Little Calumet River at Munster | 12.0 ft | 12.5 ft | above flood stage |
| **SORI2** | DuPage River at Shorewood | 15.5 ft | 16.4 ft | above flood stage |

Thorn Creek is the one to watch: at 11.31 ft it was **2.3 ft above flood stage and 1.3 ft above the
crest the River Forecast Center had projected**, still rising at about 0.34 ft/hour.

> **Gotcha — parse gauge identity and official flood stage from the NWS product text.** The IEM
> `nwsli.json` lookup **404s for these gauge IDs**, and guessing the river from the identifier gets it
> wrong: **BOLI2 was initially mislabelled "Des Plaines River at Riverside"; it is in fact the East
> Branch DuPage River at Bolingbrook.** The reliable source is the NWS Flood Warning (FLW) product
> itself, retrieved via the IEM `nwstext` API, which states for each gauge in plain text both
> `Flood stage is X ft` and `...crest of Y ft`.
>
> **A single FLW product covers several rivers**, each with its own flood stage and forecast crest
> paragraph. The product must be **split per gauge** before parsing, or every river in the county warning
> inherits the first river's numbers — a silent error that produces a table of confidently wrong flood
> stages.

In [ ]:
import re

# Authoritative gauge metadata, parsed from the NWS FLW product text via the IEM
# nwstext API. Do NOT use nwsli.json (404s for these IDs) and do NOT infer the
# river name from the gauge identifier.
GAUGE_NAME = {
    "THNI2": "Thorn Creek at Thornton",
    "BOLI2": "E. Br. DuPage River at Bolingbrook",
    "LCMI3": "Little Calumet River at Munster",
    "SORI2": "DuPage River at Shorewood",
}
FLOOD_STAGE = {"THNI2": 9.0, "BOLI2": 20.0, "LCMI3": 12.0, "SORI2": 15.5}
FORECAST_CREST = {"THNI2": 10.0, "BOLI2": 20.3, "LCMI3": 12.5, "SORI2": 16.4}


def parse_flw_segments(product_text):
    """Split an FLW product into per-gauge segments and pull stage and crest.

    A single Flood Warning covers several rivers; each has its own
    'Flood stage is X ft' and 'crest of Y ft'. Splitting first is essential.
    """
    out = []
    segments = re.split(r"\n\s*\n(?=The [A-Z])", product_text)
    for seg in segments:
        fs = re.search(r"[Ff]lood stage is ([\d.]+)\s*(?:feet|ft)", seg)
        cr = re.search(r"crest of ([\d.]+)\s*(?:feet|ft)", seg)
        river = re.match(r"\s*(The [^\n.]+)", seg)
        if fs:
            out.append(dict(river=river.group(1).strip() if river else None,
                            flood_stage_ft=float(fs.group(1)),
                            forecast_crest_ft=float(cr.group(1)) if cr else np.nan))
    return out


def hads_stage(station, network="IL_DCP",
               start="2026-07-26T00:00Z", end="2026-08-02T00:00Z"):
    """River stage (HG*) time series from the IEM HADS/DCP archive."""
    u = (f"{IEM}/hads.py?network={network}&stations={station}"
         f"&sts={start}&ets={end}&what=txt")
    d = pd.read_csv(io.StringIO(requests.get(u, timeout=120).text))
    d["utc_valid"] = pd.to_datetime(d.utc_valid, utc=True)
    d["cdt"] = d.utc_valid.dt.tz_convert("America/Chicago")
    return d


RIV = {g: hads_stage(g) for g in GAUGE_NAME}

rows = []
for g, d in RIV.items():
    col = [c for c in d.columns if c.startswith("HG")][0]   # HG = gauge height
    s = d.dropna(subset=[col]).sort_values("cdt")
    j27 = s[(s.cdt >= pd.Timestamp("2026-07-27", tz="America/Chicago")) &
            (s.cdt < pd.Timestamp("2026-07-29", tz="America/Chicago"))]
    last3 = s[s.cdt >= s.cdt.iloc[-1] - pd.Timedelta(hours=3)]
    hours = (last3.cdt.iloc[-1] - last3.cdt.iloc[0]).total_seconds() / 3600
    rate = (last3[col].iloc[-1] - last3[col].iloc[0]) / hours
    rows.append(dict(gauge=g, name=GAUGE_NAME[g],
                     flood_stage_ft=FLOOD_STAGE[g],
                     crest_27jul_ft=round(float(j27[col].max()), 2) if len(j27) else np.nan,
                     latest_ft=round(float(s[col].iloc[-1]), 2),
                     latest_time=s.cdt.iloc[-1].strftime("%b %-d %-I:%M %p CDT"),
                     forecast_crest_ft=FORECAST_CREST[g],
                     rate_ft_per_h=round(float(rate), 3),
                     above_flood_stage=bool(s[col].iloc[-1] >= FLOOD_STAGE[g])))
STATUS = pd.DataFrame(rows)
print(STATUS.to_string(index=False))


### 3.4 Figure 4 — the flooding event

Panel (a) maps HRRR storm-total precipitation for 31 July – 1 August with local storm reports overlaid on
the *same* colour scale, so the model field and the point reports are directly comparable — the point
reports sit well above the model field in the band maxima. The precipitation colormap is `ChaseSpectral`
from **cmweather**. Panel (b) is the Glen Ellyn hourly time series with the precipitable-water trace on
the right axis, showing 36 hours of steady rain in a column that never dried. Panel (c) is the regime
contrast of Section 3.1, with each pair scaled to its own maximum and the actual values printed above the
bars — a deliberate choice, since plotting 4350 J kg⁻¹ and 15 kt on a shared axis would be meaningless.
Panel (d) is the hydrograph pair showing the double crest.


![The 31 July to 1 August 2026 Chicagoland flooding event: HRRR storm-total precipitation with storm reports, the Glen Ellyn hourly rainfall and precipitable water time series, the environmental contrast against 27 July, and hydrographs showing a second crest exceeding the first](03_climatological_context_outputs/flood_overview.png)

*Figure 4. The 31 July – 1 August 2026 Chicagoland flooding event. Panel (d) shows stage relative to each
gauge's own pre-27 July baseline: both rivers exceed their 27 July crests and are still rising. Storm-total
precipitation is shaded with cmweather's ChaseSpectral colormap; point rain reports use the same scale.*


![Status update for the four warned Chicagoland river gauges: stage relative to flood stage through 1 August, and a comparison of each gauge's 27 July crest, current stage and forecast crest](03_climatological_context_outputs/flood_status_update.png)

*Figure 5. Chicagoland river flooding status. Left: stage relative to each gauge's own flood stage, with
the 27 July tornadic event and the 31 July – 1 August rainfall shaded. Right: 27 July crest (open circle),
current stage (triangle) and River Forecast Center forecast crest (tick) for each gauge, all expressed
relative to that gauge's flood stage. Thorn Creek has already passed its forecast crest.*

In [ ]:
apply_figure_style(sizes=(10, 9, 8))
COLS = {"THNI2": "#b03060", "BOLI2": "#1a7a3d", "LCMI3": "#1f4e8c", "SORI2": "#c77c1f"}

fig5, (ax5a, ax5b) = plt.subplots(
    1, 2, figsize=(14.2, 6.4), gridspec_kw=dict(width_ratios=[1.35, 1.0], wspace=0.30))

# (a) every gauge on a common axis: feet relative to its OWN flood stage
for g, d in RIV.items():
    col = [c for c in d.columns if c.startswith("HG")][0]
    s = d.dropna(subset=[col]).sort_values("cdt")
    y = s[col] - FLOOD_STAGE[g]
    t = s.cdt.dt.tz_localize(None)
    ax5a.plot(t, y, color=COLS[g], lw=1.7, label=GAUGE_NAME[g], zorder=3)
    ax5a.plot(t.iloc[-1], y.iloc[-1], "^", ms=8, color=COLS[g],
              mec="white", mew=1.0, zorder=5)
ax5a.axhline(0, color="k", lw=1.3, ls="--", zorder=4)
ax5a.text(pd.Timestamp("2026-07-26 04:00"), 0.16, "FLOOD STAGE",
          fontsize=8, fontweight="bold", color="k")
ax5a.axvspan(pd.Timestamp("2026-07-27 06:00"), pd.Timestamp("2026-07-28 12:00"),
             color="#b03060", alpha=0.08, lw=0, zorder=0)
ax5a.axvspan(pd.Timestamp("2026-07-31 12:00"), pd.Timestamp("2026-08-02 04:00"),
             color="#3f7fbf", alpha=0.10, lw=0, zorder=0)
ax5a.text(pd.Timestamp("2026-07-27 09:00"), -6.4, "tornadic\nstorms\n27 Jul",
          fontsize=7.6, color="#b03060")
ax5a.text(pd.Timestamp("2026-07-31 16:00"), -6.4, "flooding rain\n31 Jul \u2013 1 Aug",
          fontsize=7.6, color="#1f4e8c")
ax5a.set_ylabel("River stage relative to flood stage (ft)")
ax5a.set_xlabel("Date, 2026")
ax5a.xaxis.set_major_formatter(mdates.DateFormatter("%b %-d"))
ax5a.xaxis.set_major_locator(mdates.DayLocator())
ax5a.set_ylim(-7.5, 3.0)
ax5a.legend(loc="upper left", fontsize=7.8, ncol=1)
ax5a.set_title("All four warned gauges are above their 27 July crest", loc="left", pad=7)
panel_letter(ax5a, "a", dx=-0.075, dy=1.015)

# (b) 27 July crest -> now -> forecast crest, all relative to flood stage
y_pos = np.arange(len(STATUS))[::-1]
for k, r in enumerate(STATUS.itertuples()):
    yy, c = y_pos[k], COLS[r.gauge]
    ax5b.plot([r.crest_27jul_ft - r.flood_stage_ft, r.latest_ft - r.flood_stage_ft],
              [yy, yy], color=c, lw=2.0, alpha=0.45, zorder=2)
    ax5b.plot(r.crest_27jul_ft - r.flood_stage_ft, yy, "o", ms=7,
              mfc="white", mec=c, mew=1.8, zorder=4)
    ax5b.plot(r.latest_ft - r.flood_stage_ft, yy, "^", ms=10, color=c,
              mec="white", mew=1.0, zorder=5)
    ax5b.plot(r.forecast_crest_ft - r.flood_stage_ft, yy, "|", ms=15, color=c,
              mew=2.2, zorder=4)
    ax5b.text(r.latest_ft - r.flood_stage_ft + 0.12, yy + 0.22,
              f"{r.rate_ft_per_h:+.2f} ft/h", fontsize=7.4, color=c)
ax5b.axvline(0, color="k", lw=1.3, ls="--", zorder=1)
ax5b.set_yticks(y_pos)
ax5b.set_yticklabels([n.replace(" at ", "\nat ") for n in STATUS.name], fontsize=7.8)
ax5b.set_xlabel("Feet relative to flood stage")
ax5b.set_xlim(-4.0, 3.0)
ax5b.set_ylim(-0.6, len(STATUS) - 0.3)
ax5b.legend(handles=[
    Line2D([], [], marker="o", mfc="white", mec="0.3", mew=1.6, ls="", ms=7, label="27 July crest"),
    Line2D([], [], marker="^", color="0.3", ls="", ms=9, label="now"),
    Line2D([], [], marker="|", color="0.3", mew=2.2, ls="", ms=13, label="forecast crest")],
    loc="lower left", fontsize=7.6, framealpha=0.93)
ax5b.set_title("Thorn Creek has already passed its forecast crest", loc="left", pad=7)
panel_letter(ax5b, "b", dx=-0.42, dy=1.015)

stamp = STATUS.latest_time.iloc[0]
fig5.suptitle("Chicagoland river flooding \u2014 status update",
              x=0.07, ha="left", fontsize=12.5, y=1.0)
fig5.text(0.07, 0.944,
          f"Gauge observations through {stamp}. All four rivers under NWS flood warning; "
          "Thorn Creek is 2.3 ft above flood stage and still rising at 0.34 ft/h.",
          fontsize=8.5, color=META_GREY, ha="left")
fig5.subplots_adjust(top=0.85)

fig5.savefig("flood_status_update.png", dpi=200, bbox_inches="tight")


## 4. What we learned

**July 2026 was hot, and its nights were hotter than its days.** The month averaged 76.8 °F, 2.2 °F above
the 1991–2020 normal and 13th warmest of 57 Julys. But the mean overnight low ran +2.6 °F against +1.8 °F
for the daily maximum, with 12 nights at or above 70 °F. The asymmetry is the part with a health
consequence, because failure to cool overnight is what makes a heat episode dangerous.

**The month was wet in a specific, consequential way.** 5.36 inches, 9th wettest of 57 — but on only six
wet days, near-record *few*, the 9th percentile. Three days delivered an inch or more, tying the 57-year
record maximum, and the wettest three days accounted for 95 percent of the total. Rain per wet day was
0.89 inches against a normal of 0.36, second highest in 57 years. **It did not rain often; it rained
hard.**

**Exactly one July metric shows a defensible trend.** Of ten metrics tested over 1970–2026, only the mean
daily minimum temperature is significant: +0.48 °F per decade, p = 0.022. Mean temperature (p = 0.079),
total precipitation (p = 0.057), 1-inch-day counts (p = 0.092) and rain-per-wet-day (p = 0.079) all fall
short at α = 0.05. Chicago's July nights are measurably warming; for July rainfall a 57-year
single-station record cannot separate trend from variability. Several of these are close enough that the
answer is probably "yes, with more data" — but that is a different sentence from "the data show it", and
only one of the two survives a hostile question.

**The 27 July moisture was locally extreme and areawide merely moist.** Against 5,025 observed July
soundings, the corridor column of 2.29 inches sat at the 99.4th percentile, 1.7 times the July median at
Lincoln — but the domain mean was 1.83 inches, the 84th percentile, and only 3 percent of the domain
exceeded 2.25 inches. **The rarity was the combination of a 99th-percentile local column with anomalous
deep-layer shear, not the moisture alone.** High-PWAT days are ordinary in a Chicago July; 65 kt of
0–6 km shear on such a day is not.

**The flooding four days later was a different storm and a different explanation.** MLCAPE under
350 J kg⁻¹ and 0–6 km shear near 15 kt, against 4350 J kg⁻¹ and 65 kt on the 27th — but PWAT still around
1.9 inches: efficient warm-rain and stratiform processes rather than deep convection. And the crucial
finding is hydrological rather than meteorological. **The warned gauges had already flooded on 27 July,
receded for four days without returning to baseline, then exceeded their 27 July crests.** The flooding
was a second crest on ground that never dried out, and antecedent wetness — not rainfall rate — is the
load-bearing explanation. That connects directly back to the July concentration statistic: three
1-inch-plus days in one month is precisely the pattern that produces sequential flooding.

## 5. Caveats

**One station for the temperature and precipitation climatology.** O'Hare is the longest consistent
metropolitan record and the only sensible choice for a 57-year ranking, but it is a single point. It
necessarily misses the street-to-street variability that mattered in both floods — the same variability
visible in the 4 to 5.6 inch point reports against 1 to 2.3 inches at the airports.

**Trend significance here is a screening test, not an attribution study.** The regressions are ordinary
least squares on annual values, assuming independent years and a linear form, with no adjustment for
having tested ten metrics at once. Under a multiple-comparison correction even the surviving p = 0.022
would be marginal. The result identifies where to look; it does not attribute anything to a cause.

**The PWAT climatology samples the diurnal cycle coarsely.** Radiosondes are launched at 00 and 12 UTC
only, so the distribution has no information about mid-afternoon columns. Comparing a 17 UTC model field
to a 00/12 UTC observed distribution is therefore approximate, and the direction of the bias is not
obvious: 17 UTC follows the daytime moistening of the boundary layer but precedes the evening maximum.

**Chicago has no upper-air site.** The model field is placed on the nearest *upstream* distribution
(Lincoln IL, KILX, roughly 190 km southwest), which is the defensible choice for southwesterly inflow but
is not a Chicago sounding. Reasonable people could prefer a different reference site and would get
slightly different percentiles.

**HRRR is a model, including where it is used as an analysis.** The 17 UTC "analysis" fields are 1-hour
forecasts from the preceding cycle. They are the best available gridded depiction of the environment at
kilometre scale, and they are not observations.

**The flood numbers are a snapshot of a live event.** All gauge stages, rates of rise and warning statuses
were current at **9:54 pm CDT on 1 August 2026**, with three of four rivers still rising and Thorn Creek
already past its forecast crest. Anyone using these numbers after that time should re-query the gauges;
the qualitative finding (a second crest exceeding the first) is robust, but the specific stages are not.


---

Analysis by Scott Collis and Claude.